# Answer Curation

Analysis and cleanup of the results produced by `04_answer_preparation.ipynb`.

**Problem**: some passages in the top-100 have no entities identified by ReFiNed (2,888 out of 91,770 = 3.1%).  
**Goal**: determine how many queries are affected and decide whether to replace them or handle them with `kg_score = 0`.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import polars as pl
import torch

# --- Paths (same as 04_answer_preparation) ---
QUERIES_PATH = Path("data/NQ_question/qa_all_entities.jsonl")
FAISS_DIR    = Path("data/faiss_index")
OUTPUT_DIR   = Path("data/NQ_answer")
CORPUS_PATH  = Path("data/wikipedia_2018_sentence_aligned/psgs_w100_sentence.tsv")

# --- Model ---
MODEL_NAME = "facebook/contriever"
MAX_LENGTH = 512

# --- Retrieval ---
TOP_K      = 100
N_SHARDS   = 9
BATCH_SIZE = 256

# --- Candidate search ---
MAX_CANDIDATES = 5_000  # max replacement candidates to process

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Load Data

- `passage_entities.parquet`: columns `id`, `title`, `text`, `qids` (list of QIDs per passage)
- `top100_merged.parquet`: columns `query_id`, `passage_id`, `score`, `rank`, `shard_id`

In [2]:
psg_ent = pl.read_parquet(OUTPUT_DIR / "passage_entities.parquet")
top100  = pl.read_parquet(OUTPUT_DIR / "top100_merged.parquet")

print(f"Passages:  {psg_ent.height:,}")
print(f"Top-100 rows: {top100.height:,}  ({top100['query_id'].n_unique():,} queries)")
print()
print("passage_entities schema:")
print(psg_ent.schema)
print()
print("top100_merged schema:")
print(top100.schema)

Passages:  91,770
Top-100 rows: 100,000  (1,000 queries)

passage_entities schema:
Schema({'id': Int64, 'title': String, 'text': String, 'qids': List(String)})

top100_merged schema:
Schema({'query_id': Int32, 'passage_id': Int64, 'score': Float32, 'rank': Int16, 'shard_id': Int8})


## 2. Passages with 0 Entities

Identify passages where ReFiNed found no entities at all.

In [3]:
# Add entity count column
psg_ent = psg_ent.with_columns(
    pl.col("qids").list.len().alias("n_qids")
)

zero_ent = psg_ent.filter(pl.col("n_qids") == 0)
print(f"Passages with 0 entities: {zero_ent.height:,} / {psg_ent.height:,} ({zero_ent.height/psg_ent.height*100:.1f}%)")

Passages with 0 entities: 2,888 / 91,770 (3.1%)


## 3. Query Impact

How many queries have at least one 0-entity passage in their top-100?  
How many 0-entity passages does each affected query have?

In [4]:
zero_ent_ids = zero_ent.select(pl.col("id").alias("passage_id"))

# Join: which (query, passage) pairs are affected?
affected = top100.join(zero_ent_ids, on="passage_id", how="inner")
print(f"Top-100 rows affected: {affected.height:,}")

# Per-query breakdown
affected_queries = affected.group_by("query_id").agg(
    pl.col("passage_id").count().alias("n_zero_ent_passages"),
    pl.col("rank").min().alias("best_rank"),  # highest-ranked 0-ent passage
)

n_total_queries = top100["query_id"].n_unique()
n_affected = affected_queries.height
print(f"Queries affected: {n_affected:,} / {n_total_queries:,} ({n_affected/n_total_queries*100:.1f}%)")

Top-100 rows affected: 3,080
Queries affected: 344 / 1,000 (34.4%)


In [5]:
print("0-entity passages per query (affected queries only):")
print(affected_queries["n_zero_ent_passages"].describe())

print("\nBest (lowest) rank of a 0-entity passage per query:")
print(affected_queries["best_rank"].describe())

0-entity passages per query (affected queries only):
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 344.0     │
│ null_count ┆ 0.0       │
│ mean       ┆ 8.953488  │
│ std        ┆ 13.354469 │
│ min        ┆ 1.0       │
│ 25%        ┆ 1.0       │
│ 50%        ┆ 3.0       │
│ 75%        ┆ 9.0       │
│ max        ┆ 68.0      │
└────────────┴───────────┘

Best (lowest) rank of a 0-entity passage per query:
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 344.0     │
│ null_count ┆ 0.0       │
│ mean       ┆ 31.729651 │
│ std        ┆ 28.346248 │
│ min        ┆ 0.0       │
│ 25%        ┆ 7.0       │
│ 50%        ┆ 22.0      │
│ 75%        ┆ 53.0      │
│ max        ┆ 98.0      │
└────────────┴───────────┘


## 4. Decision

Based on the results above:

- **Few queries affected** (< 100) — we can replace them from the pool of 31,372 available queries
- **Many queries affected** (> 500) — replacing is not practical, better to handle with `kg_score = 0`
- **High best rank** (0-entity passages at the bottom of the ranking) — negligible impact, safe to ignore

## 5. Load Full Query Pool & Identify Candidates

Load all 31,372 queries, sort by entity count (same criterion as `04_answer_preparation` section 3b),
remove the 1,000 already selected, and take the next candidates in order.

In [6]:
# --- 1. Load all queries ---
with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    all_queries = [json.loads(line) for line in f]
print(f"Total queries: {len(all_queries):,}")

# --- 2. Count unique QIDs per query (same logic as answer_preparation 3b) ---
entity_counts = []
for i, q in enumerate(all_queries):
    qids = set(q["question_qids"])
    for variant in q["answer_variant_qids"]:
        qids.update(variant)
    entity_counts.append((i, len(qids)))

# --- 3. Sort by entity count (stable: original order preserved at equal count) ---
entity_counts.sort(key=lambda x: x[1])

# --- 4. Load the already-selected 1,000 query indices ---
subset_path = OUTPUT_DIR / "queries_subset.jsonl"
selected_orig_ids = set()
with open(subset_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        selected_orig_ids.add(record["original_query_id"])
print(f"Already selected: {len(selected_orig_ids):,}")

# --- 5. Queries needing replacement (from section 3) ---
queries_to_replace = set(affected_queries["query_id"].to_list())
print(f"Queries to replace: {len(queries_to_replace):,}")

# --- 6. Take the next candidates in order, skipping already selected ---
candidates = []
for orig_idx, n_ent in entity_counts:
    if orig_idx in selected_orig_ids:
        continue
    candidates.append((orig_idx, n_ent, all_queries[orig_idx]))
    if len(candidates) >= MAX_CANDIDATES:
        break

print(f"Candidate pool: {len(candidates):,} queries")
print(f"Entity count range: {candidates[0][1]} – {candidates[-1][1]}")

Total queries: 31,372
Already selected: 1,000
Queries to replace: 344
Candidate pool: 5,000 queries
Entity count range: 2 – 2


## 6. Encode Candidate Queries with Contriever

Same mean-pooling encoding as `04_answer_preparation` section 4 and `03_embedding.ipynb`.

In [7]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

def mean_pooling(outputs, attention_mask):
    """Mean pool over non-padding tokens (standard Contriever pooling)."""
    hidden = outputs.last_hidden_state                        # (B, S, 768)
    mask   = attention_mask.unsqueeze(-1).float()              # (B, S, 1)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1)        # (B, 768)

# --- Encode all candidate queries in batches ---
cand_texts = [c[2]["question"] for c in candidates]
all_embeddings = []

t0 = time.time()
for start in range(0, len(cand_texts), BATCH_SIZE):
    batch = cand_texts[start : start + BATCH_SIZE]
    inputs = tokenizer(
        batch, padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        emb = mean_pooling(model(**inputs), inputs["attention_mask"])
    all_embeddings.append(emb.cpu().numpy())

cand_embeddings = np.vstack(all_embeddings).astype(np.float32)
elapsed = time.time() - t0
print(f"Encoded {cand_embeddings.shape[0]:,} candidates → shape {cand_embeddings.shape} in {elapsed:.1f}s")

Encoded 5,000 candidates → shape (5000, 768) in 1.3s


## 7. Retrieve Top-100 for Candidates (all shards)

Same shard-by-shard FAISS retrieval as `04_answer_preparation` section 6-7.  
Results are saved to `top100_candidates.parquet`.

In [ ]:
import os
import sys


# WARNING — PERSONAL CONFIG: i tre path qui sotto sono hardcoded sulla
# directory miniconda della macchina dev (C:/Users/Utente/miniconda3/).
# Se installi conda in un altro prefisso o lavori da un'altra macchina,
# MODIFICA i path qui sotto. Necessario perché faiss-gpu su Windows è
# disponibile solo via conda (no pip wheel) — questa workaround linka le
# DLL e importa il package conda dal venv uv.
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/bin')
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/lib')
sys.path.insert(0, 'C:/Users/Utente/miniconda3/Lib/site-packages')

import faiss

# --- Free Contriever model from GPU (we need VRAM for FAISS) ---
del model
torch.cuda.empty_cache()

shard_frames = []
t0 = time.time()

for s in range(N_SHARDS):
    idx_path = FAISS_DIR / f"shard_{s:02d}.faiss"
    ids_path = FAISS_DIR / f"shard_{s:02d}_ids.npy"

    if not idx_path.exists():
        # Build from .npy if needed (same as 04_answer_preparation sec 5)
        npy_path = FAISS_DIR / f"shard_{s:02d}.npy"
        if not npy_path.exists():
            print(f"Shard {s:02d}: SKIPPED (no .faiss or .npy)")
            continue
        print(f"Shard {s:02d}: building index from .npy ...", end=" ", flush=True)
        vecs = np.load(str(npy_path))
        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        faiss.write_index(index, str(idx_path))
        del vecs
        print("done")
    
    print(f"Shard {s:02d}: loading index ...", end=" ", flush=True)
    index = faiss.read_index(str(idx_path))
    print(f"{index.ntotal:,} vectors", end="", flush=True)

    # Move to GPU
    res = faiss.StandardGpuResources()
    gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
    del index
    print(" on GPU")

    # Search
    scores, positions = gpu_index.search(cand_embeddings, TOP_K)

    # Map positions → passage IDs
    shard_ids = np.load(str(ids_path))

    rows = []
    for q_idx in range(len(candidates)):
        for rank_pos in range(TOP_K):
            faiss_pos = positions[q_idx, rank_pos]
            if faiss_pos < 0:
                continue  # no result at this rank
            rows.append({
                "query_id": q_idx,  # local index within candidates
                "passage_id": int(shard_ids[faiss_pos]),
                "score": float(scores[q_idx, rank_pos]),
                "rank": rank_pos,
                "shard_id": s,
            })

    shard_frames.append(pl.DataFrame(rows))
    print(f"  → {len(rows):,} rows")

    # Free GPU memory
    del gpu_index, res
    torch.cuda.empty_cache()

elapsed = time.time() - t0
print(f"\nPer-shard retrieval complete in {elapsed:.1f}s")

In [9]:
# --- Merge cross-shard results: keep global top-100 per candidate query ---
all_shards = pl.concat(shard_frames)
del shard_frames

cand_top100 = (
    all_shards
    .sort(["query_id", "score"], descending=[False, True])
    .group_by("query_id", maintain_order=True)
    .head(TOP_K)
)

# Re-assign rank within global top-100
cand_top100 = cand_top100.with_columns(
    pl.col("score")
      .rank(method="ordinal", descending=True)
      .over("query_id")
      .sub(1)
      .cast(pl.Int32)
      .alias("rank")
)

cand_top100_path = OUTPUT_DIR / "top100_candidates.parquet"
cand_top100.write_parquet(cand_top100_path)
print(f"Saved {cand_top100_path} — {cand_top100.height:,} rows, {cand_top100['query_id'].n_unique():,} queries")
print(cand_top100.head(5))

Saved data\NQ_answer\top100_candidates.parquet — 500,000 rows, 5,000 queries
shape: (5, 5)
┌──────────┬────────────┬──────────┬──────┬──────────┐
│ query_id ┆ passage_id ┆ score    ┆ rank ┆ shard_id │
│ ---      ┆ ---        ┆ ---      ┆ ---  ┆ ---      │
│ i64      ┆ i64        ┆ f64      ┆ i32  ┆ i64      │
╞══════════╪════════════╪══════════╪══════╪══════════╡
│ 0        ┆ 156845     ┆ 1.38584  ┆ 0    ┆ 0        │
│ 0        ┆ 156844     ┆ 1.23864  ┆ 1    ┆ 0        │
│ 0        ┆ 24785617   ┆ 1.225746 ┆ 2    ┆ 4        │
│ 0        ┆ 24785616   ┆ 1.202904 ┆ 3    ┆ 4        │
│ 0        ┆ 41894416   ┆ 1.200777 ┆ 4    ┆ 8        │
└──────────┴────────────┴──────────┴──────┴──────────┘


## 8. Entity-Link New Passages

Many candidate passages may already exist in `passage_entities.parquet` (overlap with the original 1,000 queries).  
We only run ReFiNed on passages not already entity-linked.

In [10]:
# --- Find which passages need entity linking ---
cand_passage_ids = set(cand_top100["passage_id"].unique().to_list())
already_linked   = set(psg_ent["id"].to_list())
new_pids         = sorted(cand_passage_ids - already_linked)

print(f"Candidate passages (unique): {len(cand_passage_ids):,}")
print(f"Already entity-linked:       {len(cand_passage_ids & already_linked):,}")
print(f"Need entity linking:         {len(new_pids):,}")

Candidate passages (unique): 399,100
Already entity-linked:       23,157
Need entity linking:         375,943


In [11]:
# --- Load passage texts from corpus (only the new ones) ---
t0 = time.time()
print(f"Loading {len(new_pids):,} passages from corpus ...", flush=True)

new_pids_set = set(new_pids)
corpus = (
    pl.scan_csv(
        CORPUS_PATH,
        separator="\t",
        schema_overrides={"id": pl.Int64, "title": pl.Utf8, "text": pl.Utf8},
    )
    .filter(pl.col("id").is_in(new_pids))
    .collect()
)

elapsed = time.time() - t0
print(f"Loaded {corpus.height:,} passages in {elapsed:.1f}s")

missing = len(new_pids) - corpus.height
if missing > 0:
    print(f"WARNING: {missing:,} passage IDs not found in corpus")

Loading 375,943 passages from corpus ...
Loaded 375,943 passages in 51.2s


In [12]:
# --- ReFiNed entity linking on new passages ---
from refined.inference.processor import Refined

refined_model = Refined.from_pretrained(
    model_name="questions_model",
    entity_set="wikipedia",
    data_dir=Path("data/refined_cache"),
)
print("ReFiNed model loaded")

ReFiNed model loaded


In [13]:
from tqdm.auto import tqdm

CHUNK_SIZE = 10_000
new_ent_dir = OUTPUT_DIR / "curation_chunks"
new_ent_dir.mkdir(exist_ok=True)

n_chunks = (corpus.height + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Passages: {corpus.height:,} — Chunks: {n_chunks} × {CHUNK_SIZE:,}")

t0 = time.time()
for chunk_idx in range(n_chunks):
    chunk_path = new_ent_dir / f"chunk_{chunk_idx:03d}.parquet"

    # Resume support: skip completed chunks
    if chunk_path.exists():
        print(f"  Chunk {chunk_idx:03d}: already exists, skipping")
        continue

    start = chunk_idx * CHUNK_SIZE
    chunk = corpus.slice(start, CHUNK_SIZE)

    results = []
    for row in tqdm(chunk.iter_rows(named=True), total=chunk.height,
                    desc=f"Chunk {chunk_idx:03d}"):
        text_input = f"{row['title']} {row['text']}"
        spans = refined_model.process_text(text_input)
        qids = []
        for span in spans:
            ent = span.predicted_entity
            if ent and ent.wikidata_entity_id:
                qids.append(ent.wikidata_entity_id)
        results.append({
            "id": row["id"],
            "title": row["title"],
            "text": row["text"],
            "qids": list(set(qids)),  # deduplicate within passage
        })

    chunk_df = pl.DataFrame(results)
    chunk_df.write_parquet(chunk_path)
    n_with = chunk_df.filter(pl.col("qids").list.len() > 0).height
    print(f"  Saved {chunk_path.name} — {chunk_df.height:,} passages, "
          f"{n_with:,} with entities ({n_with/chunk_df.height*100:.1f}%)")

elapsed = time.time() - t0
print(f"\nEntity linking complete in {elapsed:.1f}s")

Passages: 375,943 — Chunks: 38 × 10,000


Chunk 000:   0%|          | 0/10000 [00:00<?, ?it/s]

C:\Users\Utente\Documents\PycharmProjects\dl-RAG-denseAndKG\.venv\Lib\site-packages\refined\inference\processor.py:293: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Saved chunk_000.parquet — 10,000 passages, 9,451 with entities (94.5%)


Chunk 001:   0%|          | 0/10000 [00:00<?, ?it/s]

...

  Saved chunk_035.parquet — 10,000 passages, 9,836 with entities (98.4%)


Chunk 036:   0%|          | 0/10000 [00:00<?, ?it/s]

  Saved chunk_036.parquet — 10,000 passages, 9,743 with entities (97.4%)


Chunk 037:   0%|          | 0/5943 [00:00<?, ?it/s]

  Saved chunk_037.parquet — 5,943 passages, 5,842 with entities (98.3%)

Entity linking complete in 4817.6s


In [14]:
# --- Merge new entity chunks and combine with existing data ---
chunk_files = sorted(new_ent_dir.glob("chunk_*.parquet"))
if chunk_files:
    new_entities = pl.concat([pl.read_parquet(f) for f in chunk_files])
    print(f"New entity-linked passages: {new_entities.height:,}")
    # Combine with existing passage_entities
    all_psg_ent = pl.concat([psg_ent.drop("n_qids"), new_entities])
else:
    new_entities = pl.DataFrame()
    all_psg_ent = psg_ent.drop("n_qids")

all_psg_ent = all_psg_ent.with_columns(
    pl.col("qids").list.len().alias("n_qids")
)
print(f"Total entity-linked passages: {all_psg_ent.height:,}")

New entity-linked passages: 375,943
Total entity-linked passages: 467,713


## 9. Find Valid Substitutes

Iterate through candidates in order. A candidate is **valid** if all 100 of its
retrieved passages have at least 1 entity. Stop when we have enough substitutes
or exhaust the candidate pool.

In [15]:
# Build a set of passage IDs with ≥1 entity for fast lookup
has_entity = set(
    all_psg_ent.filter(pl.col("n_qids") > 0)["id"].to_list()
)

n_needed = len(queries_to_replace)
valid_subs = []     # (local_query_id, orig_idx, n_entities, query_dict)
checked = 0

for local_qid in range(len(candidates)):
    # Get the 100 passage IDs for this candidate
    pids = (
        cand_top100
        .filter(pl.col("query_id") == local_qid)
        .sort("rank")["passage_id"]
        .to_list()
    )

    checked += 1

    # Check: do ALL passages have entities?
    if all(pid in has_entity for pid in pids):
        orig_idx, n_ent, q_dict = candidates[local_qid]
        valid_subs.append((local_qid, orig_idx, n_ent, q_dict))

    # Stop early if we have enough
    if len(valid_subs) >= n_needed:
        break

print(f"Checked {checked:,} candidates")
print(f"Valid substitutes found: {len(valid_subs):,} / {n_needed:,} needed")

Checked 493 candidates
Valid substitutes found: 344 / 344 needed


## 10. Report

Summary of findings. No data is replaced — this is informational only.

In [16]:
print("=" * 60)
print("CURATION REPORT")
print("=" * 60)
print()
print(f"Original queries:          1,000")
print(f"Queries with 0-ent passages: {n_affected:,}")
print(f"Candidates checked:        {checked:,} / {len(candidates):,}")
print(f"Valid substitutes found:   {len(valid_subs):,} / {n_needed:,} needed")
print()

if len(valid_subs) >= n_needed:
    print("STATUS: All replacements found.")
elif len(valid_subs) > 0:
    print(f"STATUS: Partial — {n_needed - len(valid_subs):,} replacements still missing.")
    print(f"  Consider increasing MAX_CANDIDATES (currently {MAX_CANDIDATES:,})")
    print(f"  or handling remaining queries with kg_score = 0.")
else:
    print("STATUS: No valid substitutes found. Handle with kg_score = 0.")

print()
print("Valid substitutes (first 20):")
print(f"{'Orig idx':>10}  {'Entities':>8}  Question")
print("-" * 60)
for _, orig_idx, n_ent, q_dict in valid_subs[:20]:
    print(f"{orig_idx:>10}  {n_ent:>8}  {q_dict['question']}")

# --- Save results ---
results_path = OUTPUT_DIR / "curation_results.jsonl"
with open(results_path, "w", encoding="utf-8") as f:
    for local_qid, orig_idx, n_ent, q_dict in valid_subs:
        record = {
            "local_query_id": local_qid,
            "original_query_id": orig_idx,
            "n_unique_entities": n_ent,
            "question": q_dict["question"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
print(f"\nSaved {results_path} — {len(valid_subs):,} valid substitutes")

CURATION REPORT

Original queries:          1,000
Queries with 0-ent passages: 344
Candidates checked:        493 / 5,000
Valid substitutes found:   344 / 344 needed

STATUS: All replacements found.

Valid substitutes (first 20):
  Orig idx  Entities  Question
------------------------------------------------------------
       877         2  this artist was signed in 1952 by atlantic and brought a string of hits
       879         2  who is chief in grey's anatomy season 13
       880         2  what is the dominant religion in puerto rico
       881         2  what railroad links moscow with russia eastern coast
       883         2  who won rookie of the year and cy young award in the same year
       887         2  who plays the bear on toy story 3
       888         2  who played the priest in ryan's daughter
       890         2  who played daenerys targaryen brother in game of thrones
       893         2  who voices lois in family guy season 1
       897         2  who is the fi